In [6]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("Libraries imported successfully.")

Libraries imported successfully.


In [9]:
import os

print(os.getcwd())
print("\nFiles in processed folder:\n")

for file in os.listdir("../data/processed"):
    print(file)

c:\Users\Sanjiv\Desktop\SRM\PROJECTS\FluentIQ\notebooks

Files in processed folder:

fluentiq_dataset_v1.csv
fluentiq_scored_dataset_v1.csv


In [10]:
DATASET_PATH = "../data/processed/fluentiq_scored_dataset_v1.csv"

df = pd.read_csv(DATASET_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully.
Shape: (100, 28)


,audio_file,transcript,duration_seconds,word_count,words_per_minute,average_rms_energy,average_zero_crossing_rate,spectral_centroid,spectral_bandwidth,mfcc_1,...,mfcc_10,mfcc_11,mfcc_12,mfcc_13,fluency_score,confidence_score,clarity_score,vocabulary_score,overall_score,communication_level
0,..\data\raw\librispeech\LibriSpeech\test-clean...,HE HOPED THERE WOULD BE STEW FOR DINNER TURNIP...,10.44,28,161.00,0.034645,0.127497,1704.13,1476.58,-300.5381,...,-0.9489,-0.2123,1.6172,-5.0724,75,69.29,87.25,84,77.82,Intermediate
1,..\data\raw\librispeech\LibriSpeech\test-clean...,STUFF IT INTO YOU HIS BELLY COUNSELLED HIM,3.27,8,146.56,0.019861,0.107820,1741.59,1628.54,-364.2475,...,-2.4162,4.9232,1.5388,1.1172,90,40.00,89.22,50,69.34,Intermediate
2,..\data\raw\librispeech\LibriSpeech\test-clean...,AFTER EARLY NIGHTFALL THE YELLOW LAMPS WOULD L...,6.62,18,163.02,0.038250,0.087339,1390.03,1353.31,-303.5464,...,-1.2306,0.2887,-0.0212,-3.2952,75,76.50,91.27,54,74.43,Intermediate
3,..\data\raw\librispeech\LibriSpeech\test-clean...,HELLO BERTIE ANY GOOD IN YOUR MIND,2.68,7,156.72,0.063083,0.065348,1312.21,1378.24,-313.2429,...,-2.0147,-4.1345,-0.2080,-6.3018,90,100.00,93.47,50,85.19,Advanced
4,..\data\raw\librispeech\LibriSpeech\test-clean...,NUMBER TEN FRESH NELLY IS WAITING ON YOU GOOD ...,5.22,11,126.56,0.048197,0.096332,1548.08,1483.26,-298.8683,...,0.8973,-3.4142,1.5993,-3.0086,90,96.39,90.37,50,83.67,Advanced


In [11]:
print("=" * 60)
print("Dataset Information")
print("=" * 60)

print("\nShape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nMissing Values:")
print(df.isnull().sum())

Dataset Information

Shape: (100, 28)

Columns:
['audio_file', 'transcript', 'duration_seconds', 'word_count', 'words_per_minute', 'average_rms_energy', 'average_zero_crossing_rate', 'spectral_centroid', 'spectral_bandwidth', 'mfcc_1', 'mfcc_2', 'mfcc_3', 'mfcc_4', 'mfcc_5', 'mfcc_6', 'mfcc_7', 'mfcc_8', 'mfcc_9', 'mfcc_10', 'mfcc_11', 'mfcc_12', 'mfcc_13', 'fluency_score', 'confidence_score', 'clarity_score', 'vocabulary_score', 'overall_score', 'communication_level']

Missing Values:
audio_file                    0
transcript                    0
duration_seconds              0
word_count                    0
words_per_minute              0
average_rms_energy            0
average_zero_crossing_rate    0
spectral_centroid             0
spectral_bandwidth            0
mfcc_1                        0
mfcc_2                        0
mfcc_3                        0
mfcc_4                        0
mfcc_5                        0
mfcc_6                        0
mfcc_7                       

In [12]:
print("=" * 60)
print("Communication Level Distribution")
print("=" * 60)

print(df["communication_level"].value_counts())

print("\nPercentage Distribution")

print(
    round(
        df["communication_level"].value_counts(normalize=True) * 100,
        2
    )
)

Communication Level Distribution
communication_level
Intermediate    53
Advanced        40
Beginner         7
Name: count, dtype: int64

Percentage Distribution
communication_level
Intermediate    53.0
Advanced        40.0
Beginner         7.0
Name: proportion, dtype: float64


In [13]:
import joblib
import pandas as pd

# Load the trained classifier
classifier = joblib.load("../models/speech_level_classifier.pkl")

# Feature columns (same as in Model Training)
feature_columns = [
    'duration_seconds',
    'word_count',
    'words_per_minute',
    'average_rms_energy',
    'average_zero_crossing_rate',
    'spectral_centroid',
    'spectral_bandwidth',
    'mfcc_1', 'mfcc_2', 'mfcc_3', 'mfcc_4', 'mfcc_5',
    'mfcc_6', 'mfcc_7', 'mfcc_8', 'mfcc_9', 'mfcc_10',
    'mfcc_11', 'mfcc_12', 'mfcc_13'
]

# Get feature importance
importance = pd.DataFrame({
    "Feature": feature_columns,
    "Importance": classifier.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

importance

,Feature,Importance
0,duration_seconds,0.151340
3,average_rms_energy,0.109417
1,word_count,0.094890
2,words_per_minute,0.076081
6,spectral_bandwidth,0.066637
7,mfcc_1,0.064748
17,mfcc_11,0.060862
15,mfcc_9,0.051678
19,mfcc_13,0.035941
12,mfcc_6,0.035834


In [14]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

# Features and target
X = df[feature_columns]
y = df["communication_level"]

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

scores = cross_val_score(
    rf,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

print("Cross Validation Scores:")
print(scores)

print("\nAverage Accuracy:", round(scores.mean()*100,2), "%")

Cross Validation Scores:
[0.8  0.75 0.85 0.8  0.8 ]

Average Accuracy: 80.0 %


In [15]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

X = df[feature_columns]
y = df["communication_level"]

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 5, 10, 15],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "class_weight": [None, "balanced"]
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X, y)

print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Cross Validation Accuracy:")
print(round(grid_search.best_score_ * 100, 2), "%")

Best Parameters:
{'class_weight': 'balanced', 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}

Best Cross Validation Accuracy:
85.0 %


In [16]:
best_classifier = grid_search.best_estimator_

best_classifier.fit(X, y)

print("Improved classifier trained successfully.")

Improved classifier trained successfully.


In [17]:
import joblib
from pathlib import Path

MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(exist_ok=True)

joblib.dump(best_classifier, MODEL_DIR / "speech_level_classifier_improved.pkl")

print("Improved classifier saved successfully.")

Improved classifier saved successfully.


In [18]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

X = df[feature_columns]
y_reg = df["overall_score"]

reg_param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 5, 10, 15],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

reg_grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=reg_param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

reg_grid_search.fit(X, y_reg)

print("Best Parameters:")
print(reg_grid_search.best_params_)

print("\nBest Cross Validation R²:")
print(round(reg_grid_search.best_score_, 4))

Best Parameters:
{'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}

Best Cross Validation R²:
0.5236


In [19]:
best_regressor = reg_grid_search.best_estimator_

best_regressor.fit(X, y_reg)

print("Improved regressor trained successfully.")

Improved regressor trained successfully.


In [20]:
joblib.dump(
    best_regressor,
    "../models/communication_score_regressor_improved.pkl"
)

print("Improved regressor saved successfully.")

Improved regressor saved successfully.


In [21]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

# -------------------------
# Features & Targets
# -------------------------
X = df[feature_columns]
y_class = df["communication_level"]
y_reg = df["overall_score"]

# -------------------------
# Train-Test Split
# -------------------------
X_train, X_test, y_train_class, y_test_class = train_test_split(
    X,
    y_class,
    test_size=0.2,
    random_state=42,
    stratify=y_class
)

_, _, y_train_reg, y_test_reg = train_test_split(
    X,
    y_reg,
    test_size=0.2,
    random_state=42
)

# -------------------------
# Train Improved Models
# -------------------------
best_classifier.fit(X_train, y_train_class)
best_regressor.fit(X_train, y_train_reg)

# -------------------------
# Classification Evaluation
# -------------------------
y_pred_class = best_classifier.predict(X_test)

print("=" * 60)
print("CLASSIFICATION RESULTS")
print("=" * 60)

print(f"Accuracy: {accuracy_score(y_test_class, y_pred_class) * 100:.2f}%\n")

print(classification_report(y_test_class, y_pred_class))

print("Confusion Matrix")
print(confusion_matrix(y_test_class, y_pred_class))

# -------------------------
# Regression Evaluation
# -------------------------
y_pred_reg = best_regressor.predict(X_test)

mae = mean_absolute_error(y_test_reg, y_pred_reg)
rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))
r2 = r2_score(y_test_reg, y_pred_reg)

print("\n" + "=" * 60)
print("REGRESSION RESULTS")
print("=" * 60)

print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"R²   : {r2:.3f}")

CLASSIFICATION RESULTS
Accuracy: 75.00%

              precision    recall  f1-score   support

    Advanced       0.75      0.75      0.75         8
    Beginner       0.00      0.00      0.00         1
Intermediate       0.75      0.82      0.78        11

    accuracy                           0.75        20
   macro avg       0.50      0.52      0.51        20
weighted avg       0.71      0.75      0.73        20

Confusion Matrix
[[6 0 2]
 [0 0 1]
 [2 0 9]]

REGRESSION RESULTS
MAE  : 6.87
RMSE : 8.44
R²   : 0.083


c:\Users\Sanjiv\Desktop\SRM\PROJECTS\FluentIQ\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Sanjiv\Desktop\SRM\PROJECTS\FluentIQ\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Sanjiv\Desktop\SRM\PROJECTS\FluentIQ\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifi

In [22]:
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import cross_val_score

extra_clf = ExtraTreesClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)

extra_scores = cross_val_score(
    extra_clf,
    X,
    y_class,
    cv=5,
    scoring="accuracy"
)

print("Extra Trees CV Scores:", extra_scores)
print("Average Accuracy:", round(extra_scores.mean() * 100, 2), "%")

Extra Trees CV Scores: [0.75 0.85 0.9  0.75 0.8 ]
Average Accuracy: 81.0 %
